# Integrating a graph and its timeseries: one twin, two halves

[Tutorial 01](../01-create-a-btwin-graph/create-a-btwin-graph.ipynb) built a graph — what a
building **is**. [Tutorial 05](../05-timeseries-management/timeseries-management.ipynb) built a
table of readings — what it **did**. Neither one knows about the other, and that is not an
oversight: a graph carrying a year of quarter-hourly readings would be unqueryable, and a table
carrying a spatial hierarchy would stop being a table.

So they stay apart, and something has to join them. In BTwin that something is a
**`btwin:Document`** — a node for each database, hung off the building it meters, carrying a
property set that says where the file is, which table to read and which sensor to filter on. The
graph does not hold the readings; it holds the *directions* to them.

This notebook builds both halves, joins them, and then does the one thing neither half can do
alone: it computes a figure out of the readings, divides it by a quantity out of the graph, and
records the result back into the graph as a KPI.

**No model is involved anywhere in this notebook.** Every step below is deterministic — SPARQL you
can read, SQL you can read, and arithmetic in Python. [Tutorial
08](../08-chat-with-twin/chat-with-twin.ipynb) puts a model on top of exactly these pieces, and it
is much easier to trust it once you have seen what it is driving.

The route:

1. the estate — a small graph of three buildings and their spaces
2. the readings — one SQLite file per building per utility
3. the gap, demonstrated: what each half can and cannot answer alone
4. the join — a database as a `btwin:Document`
5. asking the graph *where would I look?*
6. from rows to files: `Tool.TwinTargets`, and the four ways it says no
7. one query, every database — compiled against each before any is read
8. the estate half: `Tool.TwinOwnerTotals`
9. a figure neither half could produce, recorded back onto the buildings
10. what a model would be given

**Prerequisites**

```bash
pip install "btwin[rdf]"
```

No API key, no network, nothing billed.

In [1]:
import random
from pathlib import Path

import pandas as pd

import btwin
from btwin import (RDF, SQL, Document, Observation, Property, PropertySet,
                   Serialization, SpatialElement, Tool)

OUTPUT = Path("output")
OUTPUT.mkdir(exist_ok=True)
DB_DIR = OUTPUT / "databases"
DB_DIR.mkdir(exist_ok=True)

# The base the graph's identifiers are minted with. Every relative '@id' resolves against it,
# so the same Turtle read from two folders still denotes the same graph.
BASE_IRI = "https://example.org/harbourside/"
TABLE = "observations"
YEAR = 2025

# The folder the graph's relative FilePath values resolve against. This notebook's own,
# since that is where it writes the databases - Tool.TwinTargets takes it explicitly rather
# than assuming the working directory, which is right only by coincidence.
BASE_PATH = Path(".")

print("BTwin", btwin.__version__)

BTwin 0.5.7


## 1. The estate

Three buildings on one campus: a library, a science block and a sports hall. One to two storeys
each, and every space carries an IFC property set with its floor area, its clear height and how
many people it seats.

The three are deliberately unlike each other. The sports hall is a single 860 m² room that uses
little electricity; the science block is small and uses a lot. An estate where every building is
the same shape makes "per square metre" an expensive way of writing "total".

Nothing here is new — this is [tutorial 01](../01-create-a-btwin-graph/create-a-btwin-graph.ipynb)
in one cell.

In [2]:
# building code -> (name, {storey code: (storey name, [(space name, use)])})
ESTATE = {
    "B1": ("Library", {
        "F0": ("Ground Floor", [("Reading Room", "study"), ("Stacks", "storage")]),
        "F1": ("First Floor",  [("Study Carrels", "study"), ("Quiet Room", "study")]),
    }),
    "B2": ("Science Block", {
        "F0": ("Ground Floor", [("Prep Room", "support"), ("Store", "storage")]),
        "F1": ("First Floor",  [("Wet Lab A", "laboratory"), ("Wet Lab B", "laboratory")]),
    }),
    "B3": ("Sports Hall", {
        "F0": ("Ground Floor", [("Main Hall", "hall"), ("Changing Rooms", "support")]),
    }),
}

# use -> (net floor area m2, clear height m, people it seats)
USES = {
    "study":      (260.0, 3.6,  90),
    "storage":    (180.0, 3.0,   6),
    "support":    ( 45.0, 2.7,   4),
    "laboratory": (140.0, 3.4,  18),
    "hall":       (860.0, 8.0, 300),
}


def SpacePSet(uid, area, height, people):
    """What the graph knows about one room, as an IFC property set."""
    pset = PropertySet.Constructor(uid, "Space Quantities")
    for name, value, quantity, unit in (
        ("NetFloorArea",  area,   "IfcAreaMeasure",   "m2"),
        ("ClearHeight",   height, "IfcLengthMeasure", "m"),
        ("OccupantCount", people, "IfcCountMeasure",  "people"),
    ):
        PropertySet.SetProperty(pset, Property.Constructor(
            name, value, propertyQuantity=quantity, propertyUnit=unit))
    return pset


estate = []
site = SpatialElement.Constructor("site", "bot:Site", "Harbourside Campus")
estate.append(site)

for code, (buildingName, storeys) in ESTATE.items():
    building = SpatialElement.Constructor(code, "bot:Building", buildingName)
    SpatialElement.SetLocationRelationship(building, linkedObject=site)
    estate.append(building)

    for storeyCode, (storeyName, spaces) in storeys.items():
        storeyUID = f"{code}-{storeyCode}"
        storey = SpatialElement.Constructor(
            storeyUID, "bot:Storey", f"{buildingName} - {storeyName}")
        SpatialElement.SetLocationRelationship(storey, linkedObject=building)
        estate.append(storey)

        for number, (spaceName, use) in enumerate(spaces, start=1):
            area, height, people = USES[use]
            spaceUID = f"{storeyUID}-S{number:02d}"
            space = SpatialElement.Constructor(spaceUID, "bot:Space", spaceName)
            SpatialElement.SetLocationRelationship(space, linkedObject=storey)
            pset = SpacePSet(f"{spaceUID}-PSET", area, height, people)
            SpatialElement.SetPSetRelationship(space, pset=pset)
            estate += [space, pset]

print(f"{len(estate)} objects: 1 site, {len(ESTATE)} buildings, "
      f"{sum(len(s) for _, s in ESTATE.values())} storeys, "
      f"{sum(len(x) for _, s in ESTATE.values() for _, x in s.values())} spaces")

29 objects: 1 site, 3 buildings, 5 storeys, 10 spaces


## 2. The readings

Twelve monthly meter readings for 2025, per building, per utility — electricity in kWh and water
in m³. Six SQLite files, in the shape `Observation.Template()` defines.

**Six files rather than two.** One database per utility would be less to carry around, but the
estate is the thing being modelled and each building owns its own meter data — which is also what
makes each file describable by exactly one document node in section 4. It is the arrangement you
actually meet: one export per building, per meter, out of one billing system.

The values are drawn from a per-meter baseline and shaped by a fixed seasonal profile. They are
**not** a function of the floor areas above — the meters were never told how big their buildings
are, and that independence is what makes section 9 a real join rather than arithmetic on two
numbers that were derived from each other in the first place.

In [3]:
UTILITIES = {
    "electricity": {
        "meterCode": "EM", "observedProperty": "ElectricityConsumption", "unit": "kWh",
        "baseline": (11_000.0, 30_000.0),
        # multiplier by calendar month, Jan..Dec: cooling in summer, lighting in winter
        "season": [1.02, 0.97, 0.94, 0.88, 0.96, 1.14, 1.31, 0.83, 1.03, 1.05, 1.01, 1.06],
    },
    "water": {
        "meterCode": "WM", "observedProperty": "WaterConsumption", "unit": "m3",
        "baseline": (150.0, 800.0),
        # follows the academic year, and collapses in August
        "season": [1.04, 1.06, 1.10, 1.00, 1.05, 0.92, 0.80, 0.31, 1.02, 1.14, 1.18, 1.16],
    },
}

rng = random.Random(20260902)
databases = {}

for code in ESTATE:
    for utility, spec in UTILITIES.items():
        sensor = f"HB-{code}-{spec['meterCode']}"          # e.g. HB-B2-EM
        low, high = spec["baseline"]
        baseline = rng.uniform(low, high) / 12.0           # this meter's own average month

        rows = [[sensor, spec["observedProperty"], spec["unit"],
                 round(baseline * spec["season"][month] * rng.uniform(0.93, 1.07), 2),
                 f"{YEAR}-{month + 1:02d}-01T00:00:00Z"]
                for month in range(12)]
        frame = pd.DataFrame(rows, columns=list(Observation.Template().columns))

        path = DB_DIR / f"HB-{code}-{utility}.db"
        Observation.SQLiteByDF(frame, str(path), TABLE, ifExists="replace")

        databases[(code, utility)] = {
            "path": path, "sensor": sensor, "rows": len(frame),
            "unit": spec["unit"], "observedProperty": spec["observedProperty"],
            "total": round(float(frame["value"].sum()), 2),
        }

print(f"{'file':<26}{'sensor':<12}{'rows':>5}{'2025 total':>15}")
for record in databases.values():
    print(f"{record['path'].name:<26}{record['sensor']:<12}{record['rows']:>5}"
          f"{record['total']:>12,.0f} {record['unit']}")

file                      sensor       rows     2025 total
HB-B1-electricity.db      HB-B1-EM       12      27,496 kWh
HB-B1-water.db            HB-B1-WM       12         693 m3
HB-B2-electricity.db      HB-B2-EM       12      22,657 kWh
HB-B2-water.db            HB-B2-WM       12         579 m3
HB-B3-electricity.db      HB-B3-EM       12      14,151 kWh
HB-B3-water.db            HB-B3-WM       12         378 m3


## 3. The gap

Before joining them, it is worth being precise about what each half can answer alone. Serialize the
estate as it stands — spaces, areas, no databases — and put the same question to both halves.

In [4]:
estateJSONLD = Serialization.JSONLDByObjects(estate)
estateGraph, _turtle = RDF.ByJSONLD(jsonld=estateJSONLD, baseIRI=BASE_IRI)

# The graph half: how big is each building? A building carries no area of its own - its spaces do.
areas = RDF.Query(estateGraph, """
    PREFIX bot:   <https://w3id.org/bot#>
    PREFIX brick: <https://brickschema.org/schema/Brick#>
    PREFIX ifc:   <https://standards.buildingsmart.org/IFC/DEV/IFC4/ADD2_TC1/OWL#>
    PREFIX rdfs:  <http://www.w3.org/2000/01/rdf-schema#>
    SELECT ?building (SUM(?area) AS ?m2) (COUNT(?space) AS ?spaces) WHERE {
      ?b a bot:Building ; rdfs:label ?building .
      ?space brick:hasLocation+ ?b ;
             ifc:HasPropertySets/ifc:HasProperties ?p .
      ?p rdfs:label "NetFloorArea" ; ifc:NominalValue ?area .
    } GROUP BY ?building ORDER BY ?building""")

print("the graph knows:")
for row in areas:
    print(f"  {row['building']:<16}{float(row['m2']):>8,.0f} m2 over {row['spaces']} spaces")

# The database half: what did that meter record? Ask the one file directly.
total = Observation.SQLiteFetch(str(databases[("B2", "electricity")]["path"]), """
    SELECT "sosa:madeBySensor" AS sensor, ROUND(SUM(value), 1) AS total, unit
    FROM observations GROUP BY "sosa:madeBySensor", unit""")

print("\none database knows:")
for row in total:
    print(f"  {row['sensor']:<16}{row['total']:>8,.0f} {row['unit']}")

the graph knows:
  Library              960 m2 over 4 spaces
  Science Block        505 m2 over 4 spaces
  Sports Hall          905 m2 over 2 spaces

one database knows:
  HB-B2-EM          22,657 kWh


Two true statements, and no way to put them in the same sentence.

The graph says *Science Block, 505 m² over 4 spaces*. The database says *HB-B2-EM, 22,657 kWh*.
Nothing anywhere connects the string `HB-B2-EM` to the node labelled `Science Block` — the
connection lives in the head of whoever named the file, and in this notebook's `databases` dict,
which is a Python variable that will not survive the kernel.

Worse, the gap is silent. You can compute `22657 / 505` by hand right now and get a number that
looks like an energy use intensity, and neither half would tell you if you had just divided the
science block's kilowatt hours by the library's floor area.

## 4. The join: a database as a document

`btwin:Document` is a node for a thing that is *about* the twin without being *in* it — a drawing,
a report, a certificate. A database is exactly that, and modelling it this way costs nothing new:
it is a node with a property set, like everything else.

The property set is what makes it worth having. Without one the node says only "a database
exists". With it, the graph can answer *where would I look, what table, which sensor* — everything
needed to open the right file and filter it to the right rows.

In [5]:
def DatabasePSet(uid, record):
    """The directions to one database, as properties on its Document node."""
    # Stored relative to this notebook's folder. A relative path keeps the graph portable;
    # Tool.TwinTargets resolves it against a basePath you supply, in section 6.
    relative = str(record["path"]).replace("\\", "/")

    pset = PropertySet.Constructor(uid, "Database Location")
    for name, value, quantity, unit in (
        ("FilePath",         relative,                    "IfcText",         None),
        ("TableName",        TABLE,                       "IfcLabel",        None),
        ("SensorID",         record["sensor"],            "IfcLabel",        None),
        ("ObservedProperty", record["observedProperty"],  "IfcLabel",        None),
        ("Unit",             record["unit"],              "IfcLabel",        None),
        ("RowCount",         record["rows"],              "IfcCountMeasure", "rows"),
        ("PeriodStart",      f"{YEAR}-01-01T00:00:00Z",   "IfcDateTime",     None),
        ("PeriodEnd",        f"{YEAR}-12-01T00:00:00Z",   "IfcDateTime",     None),
    ):
        PropertySet.SetProperty(pset, Property.Constructor(
            name, value, propertyQuantity=quantity, propertyUnit=unit))
    return pset


documents = []
buildings = {node["@id"]: node for node in estate if node.get("@type") == "bot:Building"}

for code, (buildingName, _storeys) in ESTATE.items():
    for utility in UTILITIES:
        record = databases[(code, utility)]
        documentUID = f"{code}-{utility}-db"
        document = Document.Constructor(
            documentUID, f"{buildingName} - {utility} readings {YEAR}")
        pset = DatabasePSet(f"{documentUID}-PSET", record)
        Document.SetPSet(document, pset=pset)

        # The BUILDING points at its database, not the other way round. That is the direction
        # the vocabulary permits from a bot:Building, and the direction a question about a
        # building actually walks: "this building - what has it got?"
        SpatialElement.SetRelationship(
            buildings[code], "btwin:hasDocument", linkedObject=document, validate=False)
        documents += [document, pset]

print(f"{len(documents) // 2} documents, one per database, "
      f"{len(estate) + len(documents)} objects in all")

6 documents, one per database, 41 objects in all


Now serialize the whole thing. The Turtle is written to disk and read back with `RDF.ByTTL` rather
than kept as the graph `RDF.ByJSONLD` returned, and that is not ceremony: the BTwin context
declares its own prefix as the relative `btwin#`, which resolves to an absolute IRI only when the
file is parsed against a base. A graph that has been through a file has `btwin:` spelled the same
way everywhere — which is what the SPARQL in the next section depends on.

In [6]:
jsonld = Serialization.JSONLDByObjects(estate + documents,
                                       savePath=str(OUTPUT / "harbourside.jsonld"))
_graph, _turtle = RDF.ByJSONLD(jsonld=jsonld, savePath=str(OUTPUT / "harbourside.ttl"),
                               baseIRI=BASE_IRI)
graph = RDF.ByTTL(str(OUTPUT / "harbourside.ttl"), baseIRI=BASE_IRI)

print(f"{len(graph)} triples")
print("btwin: resolves to", dict(graph.namespaces())["btwin"], "\n")

# One document and its property set, as they are actually stored
for block in graph.serialize(format="turtle").split("\n\n"):
    if "B2-electricity-db" in block:
        print(block)

470 triples
btwin: resolves to https://example.org/harbourside/btwin# 

<https://example.org/harbourside/B2-electricity-db> a btwin:Document ;
    rdfs:label "Science Block - electricity readings 2025" ;
    ifc:HasPropertySets <https://example.org/harbourside/B2-electricity-db-PSET> .
<https://example.org/harbourside/B2-electricity-db-PSET> a ifc:IfcPropertySet ;
    rdfs:label "Database Location" ;
    ifc:HasProperties [ a ifc:IfcPropertySingleValue ;
            rdfs:label "TableName" ;
            ifc:NominalValue "observations" ],
        [ a ifc:IfcPropertySingleValue ;
            rdfs:label "RowCount" ;
            ifc:NominalValue 12 ;
            ifc:Unit "rows" ],
        [ a ifc:IfcPropertySingleValue ;
            rdfs:label "PeriodStart" ;
            ifc:NominalValue "2025-01-01T00:00:00Z" ],
        [ a ifc:IfcPropertySingleValue ;
            rdfs:label "FilePath" ;
            ifc:NominalValue "output/databases/HB-B2-electricity.db" ],
        [ a ifc:IfcPropertySing

## 5. Asking the graph where to look

The join earns its keep here. This is an ordinary SPARQL query, written by hand — walk from a
building to its documents, into the property set, and read out the five properties that say how to
open the file.

The `FILTER` is what makes it a *locator* rather than a dump: `"ElectricityConsumption"` picks the
three electricity databases and leaves the three water ones shut.

In [7]:
LOCATOR = """
PREFIX btwin: <https://example.org/harbourside/btwin#>
PREFIX ifc:   <https://standards.buildingsmart.org/IFC/DEV/IFC4/ADD2_TC1/OWL#>
PREFIX rdfs:  <http://www.w3.org/2000/01/rdf-schema#>
SELECT ?owner ?ownerLabel ?filePath ?table ?sensor ?property ?unit WHERE {
  ?owner btwin:hasDocument ?document ;
         rdfs:label ?ownerLabel .
  ?document ifc:HasPropertySets ?pset .
  ?pset ifc:HasProperties [ rdfs:label "FilePath"         ; ifc:NominalValue ?filePath ] ,
                          [ rdfs:label "TableName"        ; ifc:NominalValue ?table ] ,
                          [ rdfs:label "SensorID"         ; ifc:NominalValue ?sensor ] ,
                          [ rdfs:label "ObservedProperty" ; ifc:NominalValue ?property ] ,
                          [ rdfs:label "Unit"             ; ifc:NominalValue ?unit ] .
  FILTER(?property = "ElectricityConsumption")
}
"""

located = RDF.Query(graph, LOCATOR)
print(f"{len(located)} of the {len(documents) // 2} databases match\n")
for row in located:
    print(f"  {row['ownerLabel']:<16}{row['sensor']:<10}{row['unit']:<5}{row['filePath']}")

3 of the 6 databases match



  Library         HB-B1-EM  kWh  output/databases/HB-B1-electricity.db
  Science Block   HB-B2-EM  kWh  output/databases/HB-B2-electricity.db
  Sports Hall     HB-B3-EM  kWh  output/databases/HB-B3-electricity.db


Every row carries the building **and** the file, which is the whole point: a reading that arrives
without its owner cannot be ranked against a reading from anywhere else.

## 6. From rows to files

Those rows are still only claims. The graph stores a *path*, not a file: it may be relative, it may
name something that was moved or never existed, and a query that joined one hop too many may return
the same database twice.

`Tool.TwinTargets` is where the claim is checked. No model, no network — it resolves every path
against a `basePath`, tests it on disk, deduplicates by resolved path, and hands back only what can
actually be opened.

In [8]:
targets, reason = Tool.TwinTargets(located, basePath=BASE_PATH)

print(f"{len(targets)} target(s){'  ' + reason if reason else ''}\n")
for target in targets:
    print(f"  {target['ownerLabel']:<16}{target['property']:<26}{Path(target['path']).name}")
    print(f"  {'':<16}{target['owner']}")

3 target(s)

  Library         ElectricityConsumption    HB-B1-electricity.db
                  https://example.org/harbourside/B1
  Science Block   ElectricityConsumption    HB-B2-electricity.db
                  https://example.org/harbourside/B2
  Sports Hall     ElectricityConsumption    HB-B3-electricity.db
                  https://example.org/harbourside/B3


When it cannot, it says why — and the reason is written to be *actionable*, because in tutorial 08
it is what a repair agent is handed. Here is the mistake that matters most, made on purpose: a
locator that binds `?filePath` to the document's `rdfs:label` instead of to its FilePath property.
It is a plausible query, it parses, and it returns exactly the right number of rows.

In [9]:
WRONG = LOCATOR.replace(
    '[ rdfs:label "FilePath"         ; ifc:NominalValue ?filePath ] ,', ''
).replace(
    "?document ifc:HasPropertySets ?pset .",
    "?document ifc:HasPropertySets ?pset ; rdfs:label ?filePath .")

rowsWrong = RDF.Query(graph, WRONG)
found, why = Tool.TwinTargets(rowsWrong, basePath=BASE_PATH)

print(f"{len(rowsWrong)} row(s) returned, {len(found)} usable\n")
print(why)

3 row(s) returned, 0 usable

The query returned 3 row(s) but none named a file that is there. Paths bound: 'Library - electricity readings 2025', 'Science Block - electricity readings 2025', 'Sports Hall - electricity readings 2025'. They are resolved against .. Those are titles, not paths: ?filePath was bound to the document's rdfs:label. Bind it from the property whose rdfs:label is "FilePath", using its own property block.


It named the failure — *those are titles, not paths* — and said which property to bind instead.
That specificity is deliberate: "no database found" is a dead end, "you bound the label" is a fix.

The other three refusals, on rows made up by hand so no query is needed:

In [10]:
cases = {
    "nothing named ?filePath":    [{"owner": "B1", "sensor": "HB-B1-EM"}],
    "a file that is not there":   [{"filePath": "output/databases/HB-B9-electricity.db",
                                    "ownerLabel": "Annexe"}],
    "a fan-out that is too wide": [{"filePath": f"output/databases/HB-B{n}-electricity.db",
                                    "ownerLabel": f"Building {n}"} for n in (1, 2, 3)],
}

for label, rows in cases.items():
    limit = 2 if "fan-out" in label else 25
    found, why = Tool.TwinTargets(rows, basePath=BASE_PATH, maxDatabases=limit)
    print(f"{label}:\n  {len(found)} target(s) — {why}\n")

nothing named ?filePath:
  0 target(s) — The query returned rows but bound no ?filePath, so there is no database to open. Project ?filePath, taken from the file-path property of the document's property set.

a file that is not there:
  0 target(s) — The query returned 1 row(s) but none named a file that is there. Paths bound: 'output/databases/HB-B9-electricity.db'. They are resolved against .. Check that ?filePath is bound to the path property and not to a label or an IRI.

a fan-out that is too wide:
  0 target(s) — The query selected 3 databases, over the limit of 2. Narrow it to the ones the question actually asks about.



The last one is a budget rather than a correctness check. A locator that quietly dropped its
`FILTER` would open every database in the estate and read all of them; on three files that is a
nuisance, on an estate of two hundred it is a bill.

## 7. One query, every database

The three located files share a schema — they were written by the same `Observation.SQLiteByDF`
call — so one query serves all three. That is an assumption, and the next cell tests it rather than
making it: `SQL.Validate` asks SQLite to `EXPLAIN` the query against **every** file, compiling it in
full and resolving every column, without reading a single row.

Checking all three costs nothing. It is the difference between a ranking that is silently missing a
building and one that reports which building it could not read.

In [11]:
FANOUT = """
SELECT "sosa:madeBySensor"  AS sensor,
       ROUND(SUM(value), 1) AS total,
       ROUND(AVG(value), 1) AS monthlyMean,
       COUNT(*)             AS months
FROM observations
WHERE "sosa:ObservedProperty" = 'ElectricityConsumption'
GROUP BY "sosa:madeBySensor"
"""

# Grounded on the first file: they are supposed to agree, and this is what checks it
schema = Observation.SQLiteSchemaSummary(targets[0]["path"], targets[0]["table"] or TABLE)

readable = []
for target in targets:
    checked, error = SQL.Validate(FANOUT, target["path"], 100, schema["columns"])
    print(f"  {Path(target['path']).name:<26}"
          f"{'compiled' if checked else 'REFUSED — ' + error}")
    if checked:
        readable.append(target)

  HB-B1-electricity.db      compiled
  HB-B2-electricity.db      compiled
  HB-B3-electricity.db      compiled


Then read them, and **tag every row with the building it came from**. This is the step the whole
notebook has been building towards: no file knows its own building — that fact lives in the graph,
not in the readings — so the tag is applied here, from the target the row was read through.

In [12]:
merged = []
for target in readable:
    for row in Observation.SQLiteFetch(target["path"], FANOUT):
        merged.append({
            "building": target["ownerLabel"],
            "measures": target["property"],
            "unit": target["unit"],
            **row,
        })

print(f"{'building':<16}{'sensor':<10}{'total':>12}{'unit':>6}{'months':>8}")
for row in merged:
    print(f"{row['building']:<16}{row['sensor']:<10}{row['total']:>12,.0f}"
          f"{row['unit']:>6}{row['months']:>8}")

building        sensor           total  unit  months
Library         HB-B1-EM        27,496   kWh      12
Science Block   HB-B2-EM        22,657   kWh      12
Sports Hall     HB-B3-EM        14,151   kWh      12


`22,657 kWh` has become `Science Block, 22,657 kWh` — a fact you can rank, compare and divide. It
took one SPARQL query, one SQL query and no model at all.

`Cycle.TwinQueryByPrompt` does exactly this loop internally, in `Cycle._TwinFetch`, and it leaves a
database that will not answer out rather than aborting the others: it already compiled, so a
failure at this point is the file moving or locking under you, and one unreadable building should
not cost you the answer for the rest.

## 8. The estate half

For a per-square-metre figure the readings are only half of it. `Tool.TwinOwnerTotals` supplies the
other half: every numeric property of everything located inside one node, summed.

Note what it does *not* do. It sums a floor area, and it also sums a clear height, which is
nonsense — so it reports the count it summed over beside every total and leaves the caller to
refuse the ones that are not additive. Guessing which properties add up from their names would be
wrong more quietly.

In [13]:
for target in targets:
    totals = Tool.TwinOwnerTotals(graph, target["owner"])
    things = totals.pop("", {}).get("things", 0)
    print(f"{target['ownerLabel']} — {things} space(s)")
    for name, entry in sorted(totals.items()):
        print(f"    {name:<16}{entry['sum']:>10,.1f} {entry['unit'] or '':<8}over {entry['n']}")

print("\nand as the one block a model would be shown:\n")
print(Tool.TwinEstateBlock(graph, targets))

Library — 4 space(s)
    ClearHeight           13.8 m       over 4
    NetFloorArea         960.0 m2      over 4
    OccupantCount        276.0 people  over 4
Science Block — 4 space(s)
    ClearHeight           12.5 m       over 4
    NetFloorArea         505.0 m2      over 4
    OccupantCount         46.0 people  over 4
Sports Hall — 2 space(s)
    ClearHeight           10.7 m       over 2
    NetFloorArea         905.0 m2      over 2
    OccupantCount        304.0 people  over 2

and as the one block a model would be shown:

ESTATE (what the graph records about those buildings)
  Library: 4 space(s) - ClearHeight total 13.8 m over 4, NetFloorArea total 960.0 m2 over 4, OccupantCount total 276.0 people over 4
  Science Block: 4 space(s) - ClearHeight total 12.5 m over 4, NetFloorArea total 505.0 m2 over 4, OccupantCount total 46.0 people over 4
  Sports Hall: 2 space(s) - ClearHeight total 10.7 m over 2, NetFloorArea total 905.0 m2 over 2, OccupantCount total 304.0 people over 2


## 9. A figure neither half could produce

Now the join pays for itself. Electricity use per square metre needs a number from the databases
and a number from the graph, and the result can be recorded back into the graph as a KPI hanging
off the building it describes.

`Tool.TwinKPIObjects` does that. It takes a **plan** — what to record, from which column, divided
by which property — and does every sum and division itself, in Python. In tutorial 08 the plan is
what a model writes; here it is written by hand, and the two are the same dict.

That split is the important part of the design. A model may *name* a KPI and *name* the column it
comes from, but it never produces a number. Every figure that lands in the graph is arithmetic on
retrieved data, and can be recomputed by hand from the rows in section 7 and the totals in
section 8.

In [14]:
plan = {
    "setName": "Energy Performance 2025",
    "hasBeginning": f"{YEAR}-01-01T00:00:00Z",
    "hasEnd": f"{YEAR}-12-01T00:00:00Z",
    "kpis": [
        {"name": "AnnualElectricityUse",    "column": "total", "unit": "kWh",
         "divideBy": None},
        {"name": "ElectricityUseIntensity", "column": "total", "unit": "kWh/m2",
         "divideBy": "NetFloorArea"},
    ],
}

objects, proposed, skipped = Tool.TwinKPIObjects(graph, targets, merged, plan, BASE_IRI)

print(f"{len(objects)} object(s) built{'  — skipped: ' + skipped if skipped else ''}\n")
print(f"{'building':<16}{'KPI':<26}{'value':>12} {'unit':<8}computed from")
for entry in proposed:
    source = (f"{entry['from']} / {entry['dividedBy']}" if entry["dividedBy"]
              else entry["from"])
    print(f"{entry['building']:<16}{entry['kpi']:<26}{entry['value']:>12,.2f} "
          f"{entry['unit'] or '':<8}{source}")

3 object(s) built

building        KPI                              value unit    computed from
Library         AnnualElectricityUse         27,496.20 kWh     total
Library         ElectricityUseIntensity          28.64 kWh/m2  total / NetFloorArea=960
Science Block   AnnualElectricityUse         22,656.60 kWh     total
Science Block   ElectricityUseIntensity          44.86 kWh/m2  total / NetFloorArea=505
Sports Hall     AnnualElectricityUse         14,151.40 kWh     total
Sports Hall     ElectricityUseIntensity          15.64 kWh/m2  total / NetFloorArea=905


The `computed from` column is not decoration. `ElectricityUseIntensity` for the Science Block reads
`total / NetFloorArea=505`, so a figure that looks wrong can be traced back to the two numbers
behind it without leaving the notebook — and 22,656.6 ÷ 505 = 44.87 is exactly what sections 7 and
8 printed.

The ranking has also flipped. By total, the library uses the most electricity; per square metre,
the science block uses nearly three times what the sports hall does. That reversal is the reason
the join exists, and neither half of the twin could have found it.

Now write them in. `Tool.TwinApplyObjects` serializes the KPI objects through the ordinary JSON-LD
path and merges them into the graph in place — no model-authored SPARQL update anywhere, so there
is nothing here for a validator to catch.

In [15]:
before = len(graph)
written, error = Tool.TwinApplyObjects(graph, objects, BASE_IRI, replace=True)

print(f"{written:+d} triples — {before} → {len(graph)}{'  ' + error if error else ''}")

recorded = RDF.Query(graph, """
    PREFIX btwin: <https://example.org/harbourside/btwin#>
    PREFIX eko:   <http://energy.linkeddata.es/em-kpi/ontology#>
    PREFIX ifc:   <https://standards.buildingsmart.org/IFC/DEV/IFC4/ADD2_TC1/OWL#>
    PREFIX rdfs:  <http://www.w3.org/2000/01/rdf-schema#>
    SELECT ?building ?kpi ?value ?unit WHERE {
      ?set a btwin:KPISet ; btwin:hasKPIs ?k ; eko:hasAssociatedObject ?b .
      ?b rdfs:label ?building .
      ?k rdfs:label ?kpi ; ifc:NominalValue ?value .
      OPTIONAL { ?k ifc:Unit ?unit }
    } ORDER BY ?kpi ?building""")

print(f"\nread back out of the graph — {len(recorded)} KPI(s):\n")
for row in recorded:
    print(f"  {row['kpi']:<26}{row['building']:<16}"
          f"{float(row['value']):>12,.2f} {row['unit']}")

graph.serialize(destination=str(OUTPUT / "harbourside_kpis.ttl"), format="turtle")
print("\nwritten to output/harbourside_kpis.ttl; output/harbourside.ttl is untouched")

+51 triples — 470 → 521

read back out of the graph — 6 KPI(s):

  AnnualElectricityUse      Library            27,496.20 kWh
  AnnualElectricityUse      Science Block      22,656.60 kWh
  AnnualElectricityUse      Sports Hall        14,151.40 kWh
  ElectricityUseIntensity   Library                28.64 kWh/m2
  ElectricityUseIntensity   Science Block          44.86 kWh/m2
  ElectricityUseIntensity   Sports Hall            15.64 kWh/m2



written to output/harbourside_kpis.ttl; output/harbourside.ttl is untouched


`replace=True` is what makes recording a KPI twice an update rather than a corruption. RDF adds, it
does not overwrite: recompute this next January and a KPI node would otherwise carry both figures
at once, with every query returning two rows and no way to tell which is current. Each incoming
subject is cleared first instead.

The KPIs are now ordinary graph content. They answer to SPARQL, they serialize to Turtle, and — the
part that matters in tutorial 08 — they show up in `RDF.SchemaSummary`, so the next question put to
this graph can be answered from them.

## 10. What a model would be given

Everything above was hand-written. Tutorial 08 replaces the two queries with a model, and this is
the grounding it gets: the schema, plus one block the schema cannot supply.

`RDF.SchemaSummary` describes classes, predicates and shapes. It cannot say that
`ObservedProperty` holds the literal `"ElectricityConsumption"` and not `"electricity"` — and a
locator that guesses that literal parses, runs, matches nothing, and reads exactly like an estate
with no electricity meters. `Tool.TwinPropertyBlock` closes that gap: the property names in the
graph, and the values each one actually holds.

In [16]:
schemaSummary = RDF.SchemaSummary(graph)
block = Tool.TwinPropertyBlock(graph)

print(f"{len(schemaSummary['terms'])} terms in the schema\n")
print(block)

23 terms in the schema

PROPERTY VALUES (what the properties in this graph actually hold -
                 filter on these literals, spelled exactly)
  AnnualElectricityUse: 3 numeric value(s), 14151.4 to 27496.2
  ClearHeight: 5 numeric value(s), 2.7 to 8
  ElectricityUseIntensity: 3 numeric value(s), 15.6369 to 44.8646
  FilePath: 'output/databases/HB-B1-electricity.db', 'output/databases/HB-B1-water.db', 'output/databases/HB-B2-electricity.db', 'output/databases/HB-B2-water.db', 'output/databases/HB-B3-electricity.db', 'output/databases/HB-B3-water.db'
  NetFloorArea: 5 numeric value(s), 45 to 860
  ObservedProperty: 'ElectricityConsumption', 'WaterConsumption'
  OccupantCount: 5 numeric value(s), 4 to 300
  PeriodEnd: '2025-12-01T00:00:00Z'
  PeriodStart: '2025-01-01T00:00:00Z'
  RowCount: 1 numeric value(s), 12 to 12
  SensorID: 'HB-B1-EM', 'HB-B1-WM', 'HB-B2-EM', 'HB-B2-WM', 'HB-B3-EM', 'HB-B3-WM'
  TableName: 'observations'
  Unit: 'kWh', 'm3'


Note the rule it follows: a property whose values are few and textual is *vocabulary* and gets
listed in full; one whose values are many or numeric is a *measurement* and gets a range instead.
Listing every floor area would drown the block in the one thing a filter is never written against —
and `FilePath`, listed in full, is what makes a locator writable at all.

## What you have

In `output/`:

| | |
|---|---|
| `databases/HB-*.db` | six SQLite files — one building, one utility, twelve months each |
| `harbourside.jsonld`, `harbourside.ttl` | the twin as built: the estate, and the documents that point at those six files |
| `harbourside_kpis.ttl` | the same graph with section 9's KPIs recorded on the buildings |

And a pipeline with no model anywhere in it:

| step | what does it | model? |
|---|---|---|
| where would I look? | SPARQL over the documents' property sets | no |
| is that actually a file? | `Tool.TwinTargets` | no |
| will this query run here? | `SQL.Validate`, per file, via `EXPLAIN` | no |
| what do the readings say? | `Observation.SQLiteFetch`, tagged with the owner | no |
| how big is the building? | `Tool.TwinOwnerTotals` | no |
| what is the figure? | `Tool.TwinKPIObjects` — arithmetic in Python | no |
| record it | `Tool.TwinApplyObjects` | no |

[Tutorial 08](../08-chat-with-twin/chat-with-twin.ipynb) puts a model in front of exactly this, in
exactly three places: writing the locator, writing the SQL, and naming the KPIs in the plan.
Everything in the table above stays where it is.

## Questions to try

1. **Break the join.** Rename one database file and re-run section 6. `Tool.TwinTargets` drops it
   and says which path it could not find — what does section 7's table look like then?
2. **Locate the water.** Change one literal in the `FILTER` and re-run sections 5 to 7. Which of
   the KPIs in section 9's plan still means anything?
3. **Divide by the wrong thing.** Set `"divideBy": "OccupantCount"` and read the result. The
   arithmetic is right; is the KPI?
4. **Record it twice.** Re-run section 9's write cell. `written` reports a net change of 0 — why is
   that the correct answer rather than a failure?

## Where to go next

- [`../05-timeseries-management/`](../05-timeseries-management/timeseries-management.ipynb) — the
  readings half on its own: the typed query API, the validator, and an edit under a transaction.
- [`../08-chat-with-twin/`](../08-chat-with-twin/chat-with-twin.ipynb) — the same twin, asked
  questions in English, and edited by asking.
- `myproject/create_twin.py` in this repository — the same construction at estate scale: five
  buildings, ten databases, and two interactive HTML views of the graph.